In [ ]:
import h2.connection
import h2.config

# Example 1: Creating an HTTP/2 connection
print("Creating HTTP/2 connection")

# Create connection configuration for client
config = h2.config.H2Configuration(client_side=True)
print(f"Configuration created: client_side={config.client_side}")

# Create HTTP/2 connection
conn = h2.connection.H2Connection(config=config)
print(f"Connection state: {conn.state_machine.state}")

# Initiate connection
conn.initiate_connection()
print(f"Connection initiated, state: {conn.state_machine.state}")

# Get connection data to send
data_to_send = conn.data_to_send()
print(f"Initial connection data length: {len(data_to_send)} bytes")

# Check connection settings
print(f"Local settings: {conn.local_settings}")
print(f"Remote settings: {conn.remote_settings}")

print("Example 1 completed successfully")

In [ ]:
import h2.connection
import h2.settings

# Example 2: HTTP/2 settings management
print("Demonstrating HTTP/2 settings")

# Create connection
conn = h2.connection.H2Connection()
conn.initiate_connection()

# Create custom settings
new_settings = {
    h2.settings.SettingCodes.HEADER_TABLE_SIZE: 8192,
    h2.settings.SettingCodes.ENABLE_PUSH: 0,
    h2.settings.SettingCodes.MAX_CONCURRENT_STREAMS: 100,
    h2.settings.SettingCodes.INITIAL_WINDOW_SIZE: 32768
}

print(f"Original local settings: {dict(conn.local_settings)}")

# Update settings
conn.update_settings(new_settings)
settings_data = conn.data_to_send()

print(f"Settings update data length: {len(settings_data)} bytes")
print(f"Updated local settings: {dict(conn.local_settings)}")

# Display specific setting values
for code, value in new_settings.items():
    setting_name = code.name if hasattr(code, 'name') else str(code)
    print(f"Setting {setting_name}: {value}")

print("Example 2 completed successfully")

In [ ]:
import h2.connection

# Example 3: HTTP/2 stream creation and headers
print("Creating HTTP/2 streams and sending headers")

# Create client connection
conn = h2.connection.H2Connection()
conn.initiate_connection()

# Send request headers
headers = [
    (':method', 'GET'),
    (':path', '/'),
    (':authority', 'example.com'),
    (':scheme', 'https'),
    ('user-agent', 'h2-example/1.0'),
]

stream_id = conn.send_headers(1, headers)
print(f"Used stream ID: {stream_id}")

# Get data to send
data_to_send = conn.data_to_send()
print(f"Headers data length: {len(data_to_send)} bytes")

# Check stream state
print(f"Available streams: {list(conn.streams.keys())}")
if 1 in conn.streams:
    stream = conn.streams[1]
    print(f"Stream 1 state: {stream.state_machine.state}")

print("Example 3 completed successfully")

In [ ]:
import h2.connection
import h2.events
import h2.config

# Example 4: HTTP/2 events processing
print("Processing HTTP/2 events")

# Create server connection to demonstrate receiving data
config = h2.config.H2Configuration(client_side=False)
conn = h2.connection.H2Connection(config=config)
conn.initiate_connection()

# Simulate receiving HTTP/2 connection preface and settings
connection_preface = b'PRI * HTTP/2.0\r\n\r\nSM\r\n\r\n'
settings_frame = b'\x00\x00\x00\x04\x00\x00\x00\x00\x00'  # Empty settings frame

# Process the received data
try:
    events = conn.receive_data(connection_preface + settings_frame)
    print(f"Received {len(events)} events")
    
    for i, event in enumerate(events):
        print(f"Event {i+1}: {type(event).__name__}")
        if hasattr(event, 'stream_id'):
            print(f"  Stream ID: {event.stream_id}")
        if hasattr(event, 'settings'):
            print(f"  Settings: {event.settings}")
            
except Exception as e:
    print(f"Event processing: {type(e).__name__}")
    raise

print("Example 4 completed successfully")

In [ ]:
import h2.connection

# Example 5: HTTP/2 flow control demonstration
print("Demonstrating HTTP/2 flow control")

# Create connection
conn = h2.connection.H2Connection()
conn.initiate_connection()

# Show connection is ready
print(f"Connection state: {conn.state_machine.state}")
print(f"Available streams: {list(conn.streams.keys())}")

# Send headers to create a stream (all mandatory pseudo-headers included)
headers = [
    (':method', 'POST'),
    (':path', '/upload'),
    (':authority', 'example.com'),
    (':scheme', 'https')
]
stream_id = 1
conn.send_headers(stream_id, headers)
print(f"Stream {stream_id} created with headers")

# Check stream exists and get its window
if stream_id in conn.streams:
    stream_window = conn.remote_flow_control_window(stream_id=stream_id)
    print(f"Stream {stream_id} flow control window: {stream_window}")

# Send data
data_chunk = b"Hello, HTTP/2 flow control!"
conn.send_data(stream_id, data_chunk)
print(f"Sent {len(data_chunk)} bytes of data")

print("Example 5 completed successfully")